# TRIAGE-EG Notebook 37B — External Multimodal Audit/Import V3

Fail-closed recovery path. It validates and repacks the frozen external archive as `ASR_EXTERNAL_V3_VALIDATED`; it does not run Whisper, open GT, or replace A0/S1/BTC visual assets.

In [ ]:
import os
from pathlib import Path
REPO_URL='https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git'
REPO_REF='TRIAGEEG'
REPO_DIR=Path(os.environ.get('AIC_REPO_DIR','/kaggle/working/AIC2026_TeamPTK_SGU'))
SOURCE_INPUT=Path(os.environ.get('AIC_EXTERNAL_MULTIMODAL_ROOT','/kaggle/input/datasets/irthn1311/full-873-multimodal-artifacts'))
STAGE0_INPUT=Path(os.environ.get('AIC_STAGE0_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage0-audit-bundle'))
TRIAL_INPUT=Path(os.environ.get('AIC_TRIAL_ROOT','/kaggle/input/datasets/irthn1311/thunghiem-bo-de-thi'))
E5_ONNX_INPUT=Path(os.environ['AIC_E5_ONNX_ROOT']) if os.environ.get('AIC_E5_ONNX_ROOT') else None
E5_EXACT_REVISION=os.environ.get('AIC_E5_EXACT_REVISION','03415a4be176a1620747c692ed433219fabc3def')
OUTPUT_ROOT=Path('/kaggle/working/external_multimodal_audit_v3')
OUTPUT_ZIP=Path('/kaggle/working/external_multimodal_audit_v3_bundle.zip')
print({'required_inputs':{'source_archive':str(SOURCE_INPUT),'stage0_duration_manifest':str(STAGE0_INPUT),'official_trial_package':str(TRIAL_INPUT)},'optional_input':{'pinned_e5_onnx_query_encoder':str(E5_ONNX_INPUT) if E5_ONNX_INPUT else None},'internet_required':'ONLY_FOR_GIT_CLONE_OR_EXPLICIT_REFRESH','model_download_required':False,'whisper_full_run':False,'ground_truth_opened':False,'output_zip':str(OUTPUT_ZIP),'clean_asr_zip':'/kaggle/working/external_multimodal_audit_v3/asr_external_v3_validated_bundle.zip'})


In [ ]:
import subprocess,sys
if not (REPO_DIR/'.git').is_dir(): subprocess.run(['git','clone','--branch',REPO_REF,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
subprocess.run(['git','fetch','origin',REPO_REF],cwd=REPO_DIR,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO_DIR,check=True)
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip()
sys.path.insert(0,str(REPO_DIR/'src'))
print({'source_ref':REPO_REF,'HEAD':HEAD,'checkout_mode':'DETACHED_FETCH_HEAD'})


In [ ]:
def unique_file(root,name):
    matches=sorted(Path(root).rglob(name)) if Path(root).exists() else []
    if len(matches)!=1: raise RuntimeError(f'Expected exactly one {name} under {root}; found {matches}')
    return matches[0]
SOURCE_ARCHIVE=unique_file(SOURCE_INPUT,'full_873_multimodal_artifacts.tar.gz')
VIDEO_MANIFEST=unique_file(STAGE0_INPUT,'video_manifest.jsonl')
TRIAL_ZIP=unique_file(TRIAL_INPUT,'THUNGHIEM-bo-de-thi.zip')
print({'source_archive':str(SOURCE_ARCHIVE),'video_manifest':str(VIDEO_MANIFEST),'trial_zip':str(TRIAL_ZIP)})


In [ ]:
test_env=os.environ.copy(); test_env['PYTHONPATH']=str(REPO_DIR/'src')+(os.pathsep+test_env['PYTHONPATH'] if test_env.get('PYTHONPATH') else '')
test=subprocess.run([sys.executable,'-m','pytest','tests/unit/external_multimodal_v3','tests/unit/trial_p1/test_post_bcf1_evidence.py','-q'],cwd=REPO_DIR,env=test_env,capture_output=True,text=True)
TEST_SUMMARY={'returncode':test.returncode,'stdout_tail':test.stdout.splitlines()[-20:],'stderr_tail':test.stderr.splitlines()[-20:]}
if test.returncode: raise RuntimeError(TEST_SUMMARY)
print(TEST_SUMMARY)


In [ ]:
from triage_eg.external_multimodal_v3 import run_external_multimodal_import
RESULT=run_external_multimodal_import(SOURCE_ARCHIVE,VIDEO_MANIFEST,OUTPUT_ROOT)
print({'asr':RESULT.asr_decision,'ocr':RESULT.ocr_decision,'object':RESULT.object_decision,'clean_asr_bundle':RESULT.asr_bundle})


In [ ]:
from triage_eg.trial_p1.asr_v12_loader import ASRExternalV3Loader
from triage_eg.external_multimodal_v3.trial_smoke import OnnxE5QueryEncoder,run_trial_asr_smoke
LOADER=ASRExternalV3Loader(OUTPUT_ROOT)
ENCODER=OnnxE5QueryEncoder(E5_ONNX_INPUT,exact_revision=E5_EXACT_REVISION) if E5_ONNX_INPUT else None
SMOKE=run_trial_asr_smoke(LOADER,TRIAL_ZIP,OUTPUT_ROOT,e5_query_encoder=ENCODER)
if ENCODER is None: print('E5_QUERY_SMOKE_NOT_RUN: attach a pinned local ONNX asset and set AIC_E5_ONNX_ROOT; lexical smoke remains valid')
print({'loader':LOADER.validation.as_dict(),'trial_evidence':{key:SMOKE[key] for key in ('retrieved_query_count','lexical_nonempty_query_count','e5_status','qa_evidence')}})


In [ ]:
import shutil,zipfile
shutil.make_archive(str(OUTPUT_ZIP.with_suffix('')),'zip',OUTPUT_ROOT)
with zipfile.ZipFile(OUTPUT_ZIP) as archive:
    bad=archive.testzip(); members=len(archive.infolist())
if bad is not None: raise RuntimeError(f'OUTPUT_ZIP_CRC_FAIL:{bad}')
print({'download_zip':str(OUTPUT_ZIP),'size_bytes':OUTPUT_ZIP.stat().st_size,'members':members,'crc':'PASS','whisper_full_run':False,'gt_opened':False})
